In [1]:
import pandas as pd
import os
import glob, os
from tqdm.notebook import tqdm
import json
import plotly.express as px
import geopandas as gpd
import numpy as np

%matplotlib inline
pd.set_option('display.max_columns', None)

# Datos electorales

In [2]:
data_path = "../data/raw/resultados_electorales/congreso_por_municipio/"
df = None
for file in tqdm(glob.glob(data_path + "*.xlsx")):
    df_single = pd.read_excel(file, header=1, skiprows=4)
    year = os.path.splitext(file)[0].split("/")[-1][3:7]
    month = os.path.splitext(file)[0].split("/")[-1][7:9]
    df_single["Año"] = int(year)
    df_single["Mes"] = int(month)
    df_single["añomes"] = year+month
    print("yearmonth: ", year+month)
    if df is None:
        df = df_single
    else:
        df = pd.concat([df, df_single], ignore_index=True, sort=False)
        
    # QUITAR
    break

yearmonth:  199603



In [3]:
df.columns

Index(['Nombre de Comunidad', 'Código de Provincia', 'Nombre de Provincia',
       'Código de Municipio', 'Nombre de Municipio', 'Población',
       'Número de mesas', 'Total censo electoral', 'Total votantes',
       'Votos válidos', 'Votos a candidaturas', 'Votos en blanco',
       'Votos nulos', 'PP', 'PSOE', 'IU', 'CIU', 'EAJ-PNV', 'CC', 'BNG', 'HB',
       'ERC', 'PA', 'EA', 'UV', 'LVE', 'CHA', 'UC', 'UPV-BN', 'PSM-ENE', 'CDN',
       'LV-GV', 'PRT', 'PCPE', 'PH', 'PAS', 'UPL', 'FEA', 'ICV-GORORDO', 'LV',
       'CEX', 'UM', 'TC-PNC', 'P.RIOJANO', 'PEC', 'URCL', 'NA', 'AUN', 'CR',
       'SOS', 'PREPAL', 'FREPIC', 'PSPC', 'PRCM', 'FPG', 'SIEX', 'PRV', 'PRIM',
       'FEI', 'NR', 'PICC', 'AR', 'ENV', 'PAE', 'PB', 'PCN', 'UPRA', 'UAD',
       'ADEC', 'VDPA', 'N', 'ALAS', 'ABA', 'PRGU', 'LAE', 'DSA', 'LG', 'PIZ',
       'PNM', 'POR', 'AFIT', 'CCV', 'Año', 'Mes', 'añomes'],
      dtype='object')

In [4]:
# limpiar columnas index y rellenar NA
df = df[df.columns.drop(list(df.filter(regex='Unnamed')))]
df = df.fillna(0)
# eliminar columnas con menso de x votos
#df_sum = pd.DataFrame(df.sum(axis = 0, skipna = True, numeric_only=True))
#df_sum[df_sum[0] == 0].index

In [5]:
# crear codigo ine
df["CODIGOINE"] = df["Código de Provincia"].astype(int).astype(str).str.zfill(2) + df["Código de Municipio"].astype(int).astype(str).str.zfill(3)

In [6]:
df.tail(5)

,Nombre de Comunidad,Código de Provincia,Nombre de Provincia,Código de Municipio,Nombre de Municipio,Población,Número de mesas,Total censo electoral,Total votantes,Votos válidos,Votos a candidaturas,Votos en blanco,Votos nulos,PP,PSOE,IU,CIU,EAJ-PNV,CC,BNG,HB,ERC,PA,EA,UV,LVE,CHA,UC,UPV-BN,PSM-ENE,CDN,LV-GV,PRT,PCPE,PH,PAS,UPL,FEA,ICV-GORORDO,LV,CEX,UM,TC-PNC,P.RIOJANO,PEC,URCL,NA,AUN,CR,SOS,PREPAL,FREPIC,PSPC,PRCM,FPG,SIEX,PRV,PRIM,FEI,NR,PICC,AR,ENV,PAE,PB,PCN,UPRA,UAD,ADEC,VDPA,N,ALAS,ABA,PRGU,LAE,DSA,LG,PIZ,PNM,POR,AFIT,CCV,Año,Mes,añomes,CODIGOINE
8092,Comunitat Valenciana,46,Valencia / València,262,YESA (LA) ...,288,1,241,191,190,189,1,1,140,42,5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1996,3,199603,46262
8093,Comunitat Valenciana,46,Valencia / València,263,ZARRA ...,442,1,391,340,340,339,1,0,141,188,9,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1996,3,199603,46263
8094,Comunitat Valenciana,46,Valencia / València,902,GATOVA ...,494,1,426,389,386,386,0,3,103,259,16,0,0,0,0,0,0,0,0,3,3,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1996,3,199603,46902
8095,Ciudad de Ceuta,51,Ceuta,1,CEUTA ...,71926,66,50466,32404,32169,31774,395,235,17192,11439,701,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,54,0,40,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2348,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1996,3,199603,51001
8096,Ciudad de Melilla,52,Melilla,1,MELILLA ...,63570,58,43241,27169,26925,26484,441,244,13691,11599,935,0,0,0,0,0,0,0,0,0,0,0,71,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,188,0,0,0,1996,3,199603,52001


In [7]:
df.shape

(8097, 86)

# Mapa España por munipios

In [8]:
#map_path = "../data/raw/mapa_municipios/Municipios_IGN-shp/Municipios_IGN.shp"
#geodf = gpd.read_file(map_path)
#geodf_reduced = geodf[["CODIGOINE","geometry"]]
#geojson_save_path = "../data/raw/mapa_municipios/Municipios_IGN-shp/Municipios_IGN.geojson"
#guardar
#geodf_reduced.to_file(geojson_path, driver = "GeoJSON")

In [9]:
geojson_path = "../data/raw/mapa_municipios/Municipios_IGN-shp/Municipios_IGN_reduced.geojson"

In [10]:
#cargar
with open(geojson_path) as geofile:
    geojson = json.load(geofile)

# Mapa de ganadores

In [11]:
key_columns = ['Nombre de Comunidad', 'Código de Provincia', 'Nombre de Provincia',
       'Código de Municipio', 'Nombre de Municipio', 'Población',
       'Número de mesas', 'Total censo electoral', 'Total votantes',
       'Votos válidos', 'Votos a candidaturas', 'Votos en blanco',
       'Votos nulos','Año', 'Mes', 'añomes', 'CODIGOINE']
party_columns = [x for x in df.columns if x not in key_columns]

In [12]:
df["ganador"] = df[party_columns].idxmax(axis=1)

In [13]:
df["ganador"].unique()

array(['PSOE', 'PP', 'IU', 'CHA', 'UM', 'CC', 'CIU', 'ERC', 'BNG', 'HB',
       'CDN', 'EA', 'EAJ-PNV', 'P.RIOJANO'], dtype=object)

# Gráfico

In [14]:
color_map = {'PSOE':"#EF553B", 
             'PP':"#636EFA", 
             'IU':"#2CA02C", 
             'CHA':"#B82E2E", 
             'UM':"#EECA3B", 
             'CC':"#17BECF", 
             'CIU':"#3366CC", 
             'ERC':"#FF9900", 
             'BNG':"#0DF9FF", 
             'HB':"#222A2A",
             'CDN':"#AF0038", 
             'EA':"#66AA00", 
             'EAJ-PNV':"#479B55", 
             'P.RIOJANO':"#86CE00"}

In [15]:
df_filtered = df[df["añomes"]=="199603"][["ganador", "CODIGOINE"]]

In [ ]:
fig = px.choropleth_mapbox(data_frame=df_filtered, geojson=geojson, locations='CODIGOINE', featureidkey="properties.CODIGOINE", 
                           color='ganador', mapbox_style="carto-positron", color_discrete_map=color_map,
                           center={"lat": 40.463667, "lon": -3.74922}, zoom=4)
fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})
fig.show()